# Load bronze to silver

In [478]:
import os
import pandas as pd
from datetime import datetime

## Parametros

In [479]:
bronze_path = os.path.join(
    os.curdir, "Data_Lake", "Bronze"
)
silver_path = os.path.join(
    os.curdir, "Data_Lake", "Silver"
)

In [480]:
bronze_file_name = "menstrual_cycle_bronze.parquet"

silver_file_name = "menstrual_cycle_silver.parquet"

In [481]:
bronze_file_path = os.path.join(
    bronze_path, bronze_file_name
)
silver_file_path = os.path.join(
    silver_path, silver_file_name
)

## Main

### Cargar los de datos de Bronze

In [482]:
bronze_menstrual_cycle_df = pd.read_parquet(bronze_file_path)

In [483]:
bronze_menstrual_cycle_df.head()

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
0,1.0,18.0,2.0,Moderate,54.0,Low Carb,13/11/2024 20:52,26,7.0,09/12/2024 20:52,Headache,29.28,2025-11-06 19:16:40.445350
1,1.0,18.0,2.0,Moderate,54.0,Low Carb,09/12/2024 20:52,32,5.0,10/01/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
2,1.0,18.0,2.0,Moderate,54.0,Low Carb,10/01/2025 20:52,41,7.0,20/02/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
3,1.0,18.0,2.0,Moderate,54.0,Low Carb,20/02/2025 20:52,27,3.0,19/03/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
4,1.0,18.0,2.0,Moderate,54.0,Low Carb,19/03/2025 20:52,42,5.0,30/04/2025 20:52,Cramps,29.28,2025-11-06 19:16:40.445350


### Normalización de datos

#### Nulos
De cara a tratar los valores nulos, lo primero que debemos comprobar es que campos son críticos para la tabla.

Si tenemos un "id" que usaremos para identificar a una fila de manera única, no puede ser nulo.

De ahí en adelante debemos usar nuestro propio juicio para decidir que puede ser nulo y que no, y actuar en consecuencia:

- Eliminando las filas con valores nulos en esa columna
- Imputando los valores que faltan

In [484]:
bronze_menstrual_cycle_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 895 entries, 0 to 894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   User ID                893 non-null    float64       
 1   Age                    892 non-null    float64       
 2   Stress Level           893 non-null    float64       
 3   Exercise Frequency     894 non-null    object        
 4   Sleep Hours            893 non-null    float64       
 5   Diet                   895 non-null    object        
 6   Cycle Start Date       895 non-null    object        
 7   Cycle Length           895 non-null    int64         
 8   Period Length          892 non-null    float64       
 9   Next Cycle Start Date  895 non-null    object        
 10  Symptoms               895 non-null    object        
 11  BMI                    895 non-null    float64       
 12  _BronzeTimestamp       895 non-null    datetime64[us]
dtypes: da

##### User ID

In [485]:
# compuebo los nulos de la columna User Id
bronze_menstrual_cycle_df[bronze_menstrual_cycle_df['User ID'].isnull() == True]

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
173,NaN,21.0,1.0,Low,80.0,Balanced,28/01/2025 20:52,33,7.0,02/03/2025 20:52,Headache,21.57,2025-11-06 19:16:40.445350
182,NaN,45.0,4.0,Low,75.0,Low Carb,29/03/2024 20:52,39,3.0,07/05/2024 20:52,Cramps,24.82,2025-11-06 19:16:40.445350


In [486]:
# antes de eliminar los user id nulos, buscamos posibles coincidencias de edad y bmi para identificar posibles registros nulos asiciados a personas ya registradas 
def reasignar_user_id_nulos(fila, df):
    if pd.isnull(fila['User ID']):
        coincidencia = df[(df['Age'] == fila['Age']) & 
                   (df['BMI'] == fila['BMI']) & 
                   (df['User ID'].notnull())]
        if len(coincidencia) > 0:
            return coincidencia['User ID'].iloc[0]
        else:
            return None
    else:
        return fila['User ID']

In [487]:
bronze_menstrual_cycle_df['User ID'] = bronze_menstrual_cycle_df.apply(
    lambda fila: reasignar_user_id_nulos(fila, bronze_menstrual_cycle_df),
    axis=1
)

In [488]:
# comprobamos si se han reasignado valores o si debemos ahora si eliminarlos
bronze_menstrual_cycle_df[bronze_menstrual_cycle_df['User ID'].isnull() == True]

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp


In [489]:
# comprobamos que todos ninguna fecha de comienzo de ciclo se solape y, por tanto, la reasignaciones fueron correctas
bronze_menstrual_cycle_df[bronze_menstrual_cycle_df['Age']==21]

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
172,19.0,21.0,1.0,Low,80.0,Balanced,21/12/2024 20:52,38,7.0,28/01/2025 20:52,Cramps,21.57,2025-11-06 19:16:40.445350
173,19.0,21.0,1.0,Low,80.0,Balanced,28/01/2025 20:52,33,7.0,02/03/2025 20:52,Headache,21.57,2025-11-06 19:16:40.445350
174,19.0,21.0,1.0,Low,80.0,Balanced,02/03/2025 20:52,46,4.0,17/04/2025 20:52,Mood Swings,200.00,2025-11-06 19:16:40.445350
175,19.0,21.0,1.0,Low,80.0,Balanced,17/04/2025 20:52,44,5.0,10/08/2077 20:52,Mood Swings,21.57,2025-11-06 19:16:40.445350
176,19.0,21.0,1.0,Low,80.0,Balanced,31/05/2025 20:52,42,3.0,12/07/2025 20:52,Mood Swings,21.57,2025-11-06 19:16:40.445350
178,19.0,21.0,1.0,Low,80.0,Balanced,09/08/2025 20:52,37,7.0,15/09/2025 20:52,Mood Swings,21.57,2025-11-06 19:16:40.445350
394,45.0,21.0,4.0,Moderate,66.0,Vegetarian,28/05/2023 20:52,30,3.0,27/06/2023 20:52,Headache,23.37,2025-11-06 19:16:40.445350
395,45.0,21.0,4.0,Moderate,66.0,Vegetarian,27/06/2023 20:52,27,6.0,24/07/2023 20:52,Bloating,23.37,2025-11-06 19:16:40.445350
396,45.0,21.0,4.0,Moderate,66.0,Vegetarian,24/07/2023 20:52,27,6.0,20/08/2023 20:52,Mood Swings,23.37,2025-11-06 19:16:40.445350
397,45.0,21.0,4.0,Moderate,66.0,Vegetarian,20/08/2023 20:52,25,3.0,14/09/2023 20:52,Headache,23.37,2025-11-06 19:16:40.445350


In [490]:
bronze_menstrual_cycle_df[bronze_menstrual_cycle_df['Age']==45]

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
179,20.0,45.0,4.0,Low,75.0,Low Carb,19/12/2023 20:52,29,7.0,17/01/2024 20:52,Bloating,24.82,2025-11-06 19:16:40.445350
182,20.0,45.0,4.0,Low,75.0,Low Carb,29/03/2024 20:52,39,3.0,07/05/2024 20:52,Cramps,24.82,2025-11-06 19:16:40.445350
183,20.0,45.0,4.0,Low,75.0,Low Carb,07/05/2024 20:52,36,6.0,10/08/2077 20:52,Bloating,24.82,2025-11-06 19:16:40.445350
184,20.0,45.0,4.0,Low,75.0,Low Carb,12/06/2024 20:52,47,3.0,29/07/2024 20:52,Fatigue,24.82,2025-11-06 19:16:40.445350
185,20.0,45.0,4.0,Low,75.0,Low Carb,29/07/2024 20:52,34,7.0,01/09/2024 20:52,Fatigue,24.82,2025-11-06 19:16:40.445350
230,27.0,45.0,3.0,High,63.0,Vegetarian,26/12/2024 20:52,48,3.0,12/02/2025 20:52,Bloating,18.75,2025-11-06 19:16:40.445350
231,27.0,45.0,-3.0,Low,63.0,Vegetarian,12/02/2025 20:52,46,7.0,30/03/2025 20:52,Cramps,18.75,2025-11-06 19:16:40.445350
232,27.0,45.0,3.0,High,63.0,Vegetarian,30/03/2025 20:52,25,5.0,24/04/2025 20:52,Headache,18.75,2025-11-06 19:16:40.445350
233,27.0,45.0,3.0,High,63.0,Vegetarian,24/04/2025 20:52,26,4.0,20/05/2025 20:52,Fatigue,18.75,2025-11-06 19:16:40.445350
234,27.0,45.0,3.0,High,63.0,Vegetarian,20/05/2025 20:52,46,6.0,05/07/2025 20:52,Bloating,18.75,2025-11-06 19:16:40.445350


##### Edad

In [491]:
bronze_menstrual_cycle_edades_df = bronze_menstrual_cycle_df

In [492]:
bronze_menstrual_cycle_edades_df[bronze_menstrual_cycle_df['Age'].isnull() == True]

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
177,19.0,NaN,1.0,Low,80.0,Balanced,12/07/2025 20:52,28,4.0,09/08/2025 20:52,Bloating,21.57,2025-11-06 19:16:40.445350
180,20.0,NaN,4.0,Low,75.0,Low Carb,17/01/2024 20:52,500,7.0,29/02/2024 20:52,Bloating,24.82,2025-11-06 19:16:40.445350
181,20.0,NaN,4.0,Low,75.0,Low Carb,29/02/2024 20:52,5000,5.0,29/03/2024 20:52,Bloating,24.82,2025-11-06 19:16:40.445350


In [493]:
# Edades asociadas a cada usuario
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    print(bronze_menstrual_cycle_edades_df.groupby('User ID')['Age'].unique())

User ID
1.0                  [18.0]
2.0                  [34.0]
3.0                  [22.0]
4.0                  [25.0]
5.0                  [25.0]
6.0                  [44.0]
7.0                  [75.0]
8.0                  [37.0]
9.0                  [24.0]
10.0                 [33.0]
11.0                 [20.0]
12.0                 [29.0]
13.0                 [35.0]
14.0                 [26.0]
15.0                 [39.0]
16.0                 [23.0]
17.0                [140.0]
18.0                 [20.0]
19.0            [21.0, nan]
20.0            [45.0, nan]
21.0                 [19.0]
22.0                 [33.0]
23.0                 [44.0]
24.0                 [19.0]
25.0                 [26.0]
26.0                 [28.0]
27.0                 [45.0]
28.0                 [19.0]
29.0     [27.0, 38.0, 37.0]
31.0                 [37.0]
32.0                 [25.0]
33.0                 [44.0]
34.0                 [32.0]
35.0                 [29.0]
36.0                 [31.0]
37.0        

In [494]:
def reasignar_edades_nulas(fila, df):
    if pd.isnull(fila['Age']):
        coincidencia = df[(df['User ID'] == fila['User ID']) & 
                   (df['Age'].notnull())]
        if len(coincidencia) > 0:
            return coincidencia['Age'].iloc[0]
        else:
            return None
    else:
        return fila['Age']

In [495]:
bronze_menstrual_cycle_edades_df['Age'] = bronze_menstrual_cycle_edades_df.apply(
    lambda fila: reasignar_edades_nulas(fila, bronze_menstrual_cycle_edades_df),
    axis=1
)

In [496]:
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    print(bronze_menstrual_cycle_edades_df.groupby('User ID')['Age'].unique())

User ID
1.0                  [18.0]
2.0                  [34.0]
3.0                  [22.0]
4.0                  [25.0]
5.0                  [25.0]
6.0                  [44.0]
7.0                  [75.0]
8.0                  [37.0]
9.0                  [24.0]
10.0                 [33.0]
11.0                 [20.0]
12.0                 [29.0]
13.0                 [35.0]
14.0                 [26.0]
15.0                 [39.0]
16.0                 [23.0]
17.0                [140.0]
18.0                 [20.0]
19.0                 [21.0]
20.0                 [45.0]
21.0                 [19.0]
22.0                 [33.0]
23.0                 [44.0]
24.0                 [19.0]
25.0                 [26.0]
26.0                 [28.0]
27.0                 [45.0]
28.0                 [19.0]
29.0     [27.0, 38.0, 37.0]
31.0                 [37.0]
32.0                 [25.0]
33.0                 [44.0]
34.0                 [32.0]
35.0                 [29.0]
36.0                 [31.0]
37.0        

##### Stress level

In [497]:
bronze_menstrual_cycle_stresslevel_df = bronze_menstrual_cycle_edades_df

In [498]:
bronze_menstrual_cycle_stresslevel_df[bronze_menstrual_cycle_stresslevel_df['Stress Level'].isnull() == True]

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
38,4.0,25.0,NaN,Moderate,83.0,Vegetarian,08/02/2025 20:52,29,5.0,09/03/2025 20:52,Fatigue,25.68,2025-11-06 19:16:40.445350
53,6.0,44.0,NaN,Low,61.0,Low Carb,03/06/2023 20:52,39,NaN,12/07/2023 20:52,Mood Swings,25.87,2025-11-06 19:16:40.445350


In [499]:
# Rellenamos los valores nulos con la media de valores de Stress level del usuario correspondiente (User ID).
bronze_menstrual_cycle_stresslevel_df['Stress Level'] = bronze_menstrual_cycle_stresslevel_df.groupby('User ID')['Stress Level'].transform(
    lambda x: x.fillna(x.mean())
)

##### Exercise frecuency

In [500]:
bronze_menstrual_cycle_exercisefrequency_df = bronze_menstrual_cycle_stresslevel_df

In [501]:
# Siendo un unico valor nulo, no añadimos una categoria mas "desconcocida" ya que a la hora de analizar perdemos agrupacion
# Rellenamos el nulo con la categoria mas repetida o moda
fill_values = {
    "Exercise Frequency": bronze_menstrual_cycle_exercisefrequency_df['Exercise Frequency'].mode()[0]
}
# [0] al final por si hay un empate que escoja el primer valor categorico mas repetido

In [502]:
# Aplico la limpieza
bronze_menstrual_cycle_exercisefrequency_df = bronze_menstrual_cycle_exercisefrequency_df.fillna(fill_values)

In [503]:
bronze_menstrual_cycle_exercisefrequency_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 895 entries, 0 to 894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   User ID                895 non-null    float64       
 1   Age                    895 non-null    float64       
 2   Stress Level           895 non-null    float64       
 3   Exercise Frequency     895 non-null    object        
 4   Sleep Hours            893 non-null    float64       
 5   Diet                   895 non-null    object        
 6   Cycle Start Date       895 non-null    object        
 7   Cycle Length           895 non-null    int64         
 8   Period Length          892 non-null    float64       
 9   Next Cycle Start Date  895 non-null    object        
 10  Symptoms               895 non-null    object        
 11  BMI                    895 non-null    float64       
 12  _BronzeTimestamp       895 non-null    datetime64[us]
dtypes: da

##### Period length & Sleep hours

In [504]:
bronze_menstrual_cycle_periodlength_df = bronze_menstrual_cycle_exercisefrequency_df

In [505]:
# las filas asociadas a las variables numericas duracion del periodo y horas dormidas serán rellenadas con la media de duraciones de dicho usuario
bronze_menstrual_cycle_periodlength_df['Period Length'] = bronze_menstrual_cycle_periodlength_df.groupby('User ID')['Period Length'].transform(lambda x:x.fillna(x.mean()))

In [506]:
bronze_menstrual_cycle_sleephours_df = bronze_menstrual_cycle_periodlength_df

In [507]:
bronze_menstrual_cycle_sleephours_df['Sleep Hours'] = bronze_menstrual_cycle_sleephours_df.groupby('User ID')['Sleep Hours'].transform(lambda x:x.fillna(x.mean()))

In [508]:
bronze_menstrual_cycle_sleephours_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 895 entries, 0 to 894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   User ID                895 non-null    float64       
 1   Age                    895 non-null    float64       
 2   Stress Level           895 non-null    float64       
 3   Exercise Frequency     895 non-null    object        
 4   Sleep Hours            895 non-null    float64       
 5   Diet                   895 non-null    object        
 6   Cycle Start Date       895 non-null    object        
 7   Cycle Length           895 non-null    int64         
 8   Period Length          895 non-null    float64       
 9   Next Cycle Start Date  895 non-null    object        
 10  Symptoms               895 non-null    object        
 11  BMI                    895 non-null    float64       
 12  _BronzeTimestamp       895 non-null    datetime64[us]
dtypes: da

In [509]:
bronze_menstrual_cycle_sleephours_df[bronze_menstrual_cycle_sleephours_df['Next Cycle Start Date'].isnull() == True]

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp


#### Duplicados

In [510]:
# User ID en este caso puede, y debe, estar duplicada ya que hay varios registros por persona para analizar 
# varios ciclos menstruales

In [511]:
# Las unicas columnas criticas en conjunto son User ID y Cycle start date. Si un mismo user tiene dos fechas de inicio
# de ciclo duplicadas, no tendría sentido (al igual que fechas de fin de periodo).
duplicados = bronze_menstrual_cycle_sleephours_df.duplicated(subset=['User ID', 'Cycle Start Date'], keep=False)
bronze_menstrual_cycle_sleephours_df[duplicados].sort_values(['User ID', 'Cycle Start Date'])

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp


In [512]:
# comprobamos si hay varias edades asociadas a un mismo user
# Si esas edades son contiguas puede tener sentido. Si no, las filas son eliminadas
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    print(bronze_menstrual_cycle_sleephours_df.groupby('User ID')['Age'].unique())

User ID
1.0                  [18.0]
2.0                  [34.0]
3.0                  [22.0]
4.0                  [25.0]
5.0                  [25.0]
6.0                  [44.0]
7.0                  [75.0]
8.0                  [37.0]
9.0                  [24.0]
10.0                 [33.0]
11.0                 [20.0]
12.0                 [29.0]
13.0                 [35.0]
14.0                 [26.0]
15.0                 [39.0]
16.0                 [23.0]
17.0                [140.0]
18.0                 [20.0]
19.0                 [21.0]
20.0                 [45.0]
21.0                 [19.0]
22.0                 [33.0]
23.0                 [44.0]
24.0                 [19.0]
25.0                 [26.0]
26.0                 [28.0]
27.0                 [45.0]
28.0                 [19.0]
29.0     [27.0, 38.0, 37.0]
31.0                 [37.0]
32.0                 [25.0]
33.0                 [44.0]
34.0                 [32.0]
35.0                 [29.0]
36.0                 [31.0]
37.0        

In [513]:
# El user nº 29 tiene 3 edades distintas. Una de ellas muy alejada de lo real (27 años)
# Eliminamos dichas coincidencias
bronze_menstrual_cycle_variasedades_df = bronze_menstrual_cycle_sleephours_df[~(
    (bronze_menstrual_cycle_sleephours_df['Age'] == 27)
    &
    (bronze_menstrual_cycle_sleephours_df['User ID'] == 29))
]

In [514]:
with pd.option_context('display.max_rows',None,'display.max_columns',None):
    print(bronze_menstrual_cycle_variasedades_df.groupby('User ID')['Age'].unique())

User ID
1.0            [18.0]
2.0            [34.0]
3.0            [22.0]
4.0            [25.0]
5.0            [25.0]
6.0            [44.0]
7.0            [75.0]
8.0            [37.0]
9.0            [24.0]
10.0           [33.0]
11.0           [20.0]
12.0           [29.0]
13.0           [35.0]
14.0           [26.0]
15.0           [39.0]
16.0           [23.0]
17.0          [140.0]
18.0           [20.0]
19.0           [21.0]
20.0           [45.0]
21.0           [19.0]
22.0           [33.0]
23.0           [44.0]
24.0           [19.0]
25.0           [26.0]
26.0           [28.0]
27.0           [45.0]
28.0           [19.0]
29.0     [38.0, 37.0]
31.0           [37.0]
32.0           [25.0]
33.0           [44.0]
34.0           [32.0]
35.0           [29.0]
36.0           [31.0]
37.0           [44.0]
38.0           [45.0]
39.0           [23.0]
40.0           [18.0]
41.0           [33.0]
42.0           [24.0]
43.0           [28.0]
44.0           [26.0]
45.0           [21.0]
46.0           [25.0]
47

#### Valores atipicos

##### User ID

In [515]:
sorted(bronze_menstrual_cycle_variasedades_df['User ID'])

[1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 1.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 2.0,
 3.0,
 3.0,
 3.0,
 3.0,
 3.0,
 3.0,
 3.0,
 3.0,
 3.0,
 3.0,
 4.0,
 4.0,
 4.0,
 4.0,
 4.0,
 4.0,
 4.0,
 4.0,
 4.0,
 4.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 5.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 6.0,
 7.0,
 7.0,
 7.0,
 7.0,
 7.0,
 7.0,
 8.0,
 8.0,
 8.0,
 8.0,
 8.0,
 8.0,
 8.0,
 8.0,
 8.0,
 8.0,
 8.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 9.0,
 10.0,
 10.0,
 10.0,
 10.0,
 10.0,
 10.0,
 11.0,
 11.0,
 11.0,
 11.0,
 11.0,
 11.0,
 11.0,
 11.0,
 11.0,
 11.0,
 12.0,
 12.0,
 12.0,
 12.0,
 12.0,
 12.0,
 12.0,
 12.0,
 12.0,
 12.0,
 12.0,
 13.0,
 13.0,
 13.0,
 13.0,
 13.0,
 13.0,
 13.0,
 13.0,
 13.0,
 13.0,
 13.0,
 14.0,
 14.0,
 14.0,
 14.0,
 14.0,
 14.0,
 14.0,
 14.0,
 15.0,
 15.0,
 15.0,
 15.0,
 15.0,
 15.0,
 15.0,
 16.0,
 16.0,
 16.0,
 16.0,
 16.0,
 16.0,
 16.0,
 16.0,
 17.0,
 17.0,
 17.0,
 17.0,
 17.0

##### Edad

In [516]:
# Comporbamos valores fuera de rango en la columna edad.
# La menstruación generalmente comienza a los 16-20 años, y finaliza en la edad de la menopausia, que ocurre a los 45-55 años.
bronze_menstrual_cycle_variasedades_df['Age'].value_counts()

Age
44.0     63
25.0     63
20.0     51
26.0     46
39.0     45
35.0     43
37.0     40
45.0     40
34.0     39
31.0     37
21.0     35
22.0     34
19.0     33
18.0     33
33.0     30
29.0     29
32.0     23
38.0     23
40.0     23
24.0     22
28.0     22
27.0     18
42.0     17
36.0     15
30.0     15
23.0     14
140.0    10
41.0      8
75.0      6
43.0      6
Name: count, dtype: int64

In [517]:
# eliminamos todos los registros asociados a edades erroneas
bronze_menstrual_cycle_sin_atipicos_df = bronze_menstrual_cycle_variasedades_df[bronze_menstrual_cycle_df['Age'] <= 55]

C:\Users\cadav\AppData\Local\Temp\ipykernel_11152\882517079.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  bronze_menstrual_cycle_sin_atipicos_df = bronze_menstrual_cycle_variasedades_df[bronze_menstrual_cycle_df['Age'] <= 55]


In [518]:
bronze_menstrual_cycle_sin_atipicos_df['Age'].value_counts()

Age
25.0    63
44.0    63
20.0    51
26.0    46
39.0    45
35.0    43
45.0    40
37.0    40
34.0    39
31.0    37
21.0    35
22.0    34
19.0    33
18.0    33
33.0    30
29.0    29
40.0    23
32.0    23
38.0    23
24.0    22
28.0    22
27.0    18
42.0    17
36.0    15
30.0    15
23.0    14
41.0     8
43.0     6
Name: count, dtype: int64

##### Stress level

In [519]:
bronze_menstrual_cycle_sin_atipicos_df['Stress Level'].value_counts()

Stress Level
 4.0      226
 1.0      183
 2.0      158
 5.0      156
 3.0      142
 200.0      1
-3.0        1
Name: count, dtype: int64

In [520]:
bronze_menstrual_cycle_abs_df = bronze_menstrual_cycle_sin_atipicos_df

In [521]:
bronze_menstrual_cycle_abs_df['Stress Level']=bronze_menstrual_cycle_abs_df['Stress Level'].abs()

C:\Users\cadav\AppData\Local\Temp\ipykernel_11152\2185569480.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bronze_menstrual_cycle_abs_df['Stress Level']=bronze_menstrual_cycle_abs_df['Stress Level'].abs()


In [522]:
bronze_menstrual_cycle_abs_df['Stress Level'].value_counts()

Stress Level
4.0      226
1.0      183
2.0      158
5.0      156
3.0      143
200.0      1
Name: count, dtype: int64

In [523]:
# Eliminamos todos los registros mayores a 5 (maximo de la escala)
bronze_menstrual_cycle_stresslevel2_df = bronze_menstrual_cycle_abs_df[bronze_menstrual_cycle_abs_df['Stress Level'] <= 5]

In [524]:
bronze_menstrual_cycle_stresslevel2_df['Stress Level'].value_counts()

Stress Level
4.0    226
1.0    183
2.0    158
5.0    156
3.0    143
Name: count, dtype: int64

##### BMI

In [525]:
bronze_menstrual_cycle_stresslevel2_df['BMI'].value_counts()

BMI
24.41     17
28.82     15
25.52     12
25.45     12
25.27     12
          ..
21.47      6
20.02      6
20.99      6
222.00     1
200.00     1
Name: count, Length: 97, dtype: int64

In [526]:
# Eliminamos todos los registros mayores a 80 (maximo valor en casos medicos extremos)
bronze_menstrual_cycle_BMI_df = bronze_menstrual_cycle_stresslevel2_df[bronze_menstrual_cycle_stresslevel2_df['BMI'] <= 80]

In [527]:
bronze_menstrual_cycle_BMI_df['BMI'].value_counts()

BMI
24.41    17
28.82    15
25.52    12
24.64    12
18.83    12
         ..
21.47     6
18.59     6
23.97     6
20.02     6
20.99     6
Name: count, Length: 95, dtype: int64

##### Period length

In [528]:
bronze_menstrual_cycle_BMI_df['Period Length'].value_counts()

Period Length
5.000000    189
4.000000    175
6.000000    173
7.000000    168
3.000000    157
5.625000      1
4.111111      1
Name: count, dtype: int64

##### Sleep hours

In [529]:
sorted(bronze_menstrual_cycle_BMI_df['Sleep Hours'])

[51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 51.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 52.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 53.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,
 54.0,

##### Cycle length

In [530]:
sorted(bronze_menstrual_cycle_BMI_df['Cycle Length'])

[25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 31,
 31,
 31,
 31,


In [531]:
# Eliminamos todos los registros con valores extremos
# Dejando un margen considerable de duracion (Ej: retrasos), eliminamos los valores >50
bronze_menstrual_cycle_cyclelength_df = bronze_menstrual_cycle_BMI_df[bronze_menstrual_cycle_BMI_df['Cycle Length'] <= 50]

In [532]:
sorted(bronze_menstrual_cycle_cyclelength_df['Cycle Length'])

[25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 25,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 26,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 27,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 28,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 29,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 30,
 31,
 31,
 31,
 31,


#### Tipado

In [533]:
bronze_menstrual_cycle_cyclelength_df['Exercise Frequency'].value_counts()

Exercise Frequency
High        337
Moderate    274
Low         251
Name: count, dtype: int64

In [534]:
bronze_menstrual_cycle_cyclelength_df['Diet'].value_counts()

Diet
Vegetarian    266
Balanced      249
Low Carb      186
High Sugar    159
BALANCED        1
BL              1
Name: count, dtype: int64

In [535]:
def normalizar_dietas(valor: str)->str:
    valores_balanced = ["Balanced","BALANCED","BL"]
    if valor in valores_balanced:
        return "Balanced"
    else:
        return valor

In [536]:
bronze_menstrual_cycle_diet_df = bronze_menstrual_cycle_cyclelength_df

In [537]:
bronze_menstrual_cycle_diet_df['Diet'] = bronze_menstrual_cycle_diet_df['Diet'].apply(normalizar_dietas)

C:\Users\cadav\AppData\Local\Temp\ipykernel_11152\3485568097.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bronze_menstrual_cycle_diet_df['Diet'] = bronze_menstrual_cycle_diet_df['Diet'].apply(normalizar_dietas)


In [538]:
bronze_menstrual_cycle_diet_df['Diet'].value_counts()

Diet
Vegetarian    266
Balanced      251
Low Carb      186
High Sugar    159
Name: count, dtype: int64

In [539]:
bronze_menstrual_cycle_diet_df['Symptoms'].value_counts()

Symptoms
Bloating       195
Fatigue        172
Headache       170
Cramps         168
Mood Swings    157
Name: count, dtype: int64

In [540]:
bronze_menstrual_cycle_diet_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 862 entries, 0 to 894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   User ID                862 non-null    float64       
 1   Age                    862 non-null    float64       
 2   Stress Level           862 non-null    float64       
 3   Exercise Frequency     862 non-null    object        
 4   Sleep Hours            862 non-null    float64       
 5   Diet                   862 non-null    object        
 6   Cycle Start Date       862 non-null    object        
 7   Cycle Length           862 non-null    int64         
 8   Period Length          862 non-null    float64       
 9   Next Cycle Start Date  862 non-null    object        
 10  Symptoms               862 non-null    object        
 11  BMI                    862 non-null    float64       
 12  _BronzeTimestamp       862 non-null    datetime64[us]
dtypes: datetim

In [541]:
bronze_menstrual_cycle_diet_df.head()

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
0,1.0,18.0,2.0,Moderate,54.0,Low Carb,13/11/2024 20:52,26,7.0,09/12/2024 20:52,Headache,29.28,2025-11-06 19:16:40.445350
1,1.0,18.0,2.0,Moderate,54.0,Low Carb,09/12/2024 20:52,32,5.0,10/01/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
2,1.0,18.0,2.0,Moderate,54.0,Low Carb,10/01/2025 20:52,41,7.0,20/02/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
3,1.0,18.0,2.0,Moderate,54.0,Low Carb,20/02/2025 20:52,27,3.0,19/03/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
4,1.0,18.0,2.0,Moderate,54.0,Low Carb,19/03/2025 20:52,42,5.0,30/04/2025 20:52,Cramps,29.28,2025-11-06 19:16:40.445350


In [542]:
bronze_menstrual_cycle_diet_df['Cycle Start Date'] = pd.to_datetime(bronze_menstrual_cycle_diet_df['Cycle Start Date'])

C:\Users\cadav\AppData\Local\Temp\ipykernel_11152\64442064.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  bronze_menstrual_cycle_diet_df['Cycle Start Date'] = pd.to_datetime(bronze_menstrual_cycle_diet_df['Cycle Start Date'])
C:\Users\cadav\AppData\Local\Temp\ipykernel_11152\64442064.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bronze_menstrual_cycle_diet_df['Cycle Start Date'] = pd.to_datetime(bronze_menstrual_cycle_diet_df['Cycle Start Date'])


In [543]:
bronze_menstrual_cycle_diet_df.head()

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
0,1.0,18.0,2.0,Moderate,54.0,Low Carb,2024-11-13 20:52:00,26,7.0,09/12/2024 20:52,Headache,29.28,2025-11-06 19:16:40.445350
1,1.0,18.0,2.0,Moderate,54.0,Low Carb,2024-12-09 20:52:00,32,5.0,10/01/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
2,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-01-10 20:52:00,41,7.0,20/02/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
3,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-02-20 20:52:00,27,3.0,19/03/2025 20:52,Fatigue,29.28,2025-11-06 19:16:40.445350
4,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-03-19 20:52:00,42,5.0,30/04/2025 20:52,Cramps,29.28,2025-11-06 19:16:40.445350


In [544]:
sorted(bronze_menstrual_cycle_diet_df['Cycle Start Date'])

[Timestamp('2023-03-20 20:52:00'),
 Timestamp('2023-03-22 20:52:00'),
 Timestamp('2023-03-25 20:52:00'),
 Timestamp('2023-03-29 20:52:00'),
 Timestamp('2023-04-16 20:52:00'),
 Timestamp('2023-04-16 20:52:00'),
 Timestamp('2023-04-17 20:52:00'),
 Timestamp('2023-04-21 20:52:00'),
 Timestamp('2023-04-22 20:52:00'),
 Timestamp('2023-04-24 20:52:00'),
 Timestamp('2023-04-29 20:52:00'),
 Timestamp('2023-05-01 20:52:00'),
 Timestamp('2023-05-01 20:52:00'),
 Timestamp('2023-05-04 20:52:00'),
 Timestamp('2023-05-06 20:52:00'),
 Timestamp('2023-05-12 20:52:00'),
 Timestamp('2023-05-17 20:52:00'),
 Timestamp('2023-05-22 20:52:00'),
 Timestamp('2023-05-23 20:52:00'),
 Timestamp('2023-05-28 20:52:00'),
 Timestamp('2023-05-29 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-04 20:52:00'),
 Timestamp('2023-06-06 20:52:00'),
 Timestamp('2023-06-09 20:52:00'),
 Timestamp('2023-06-10 20:52:00'),
 Timestamp('2023-06-21 20:52:00'),
 Timestamp('2023-06-

In [556]:
bronze_menstrual_cycle_startdate_df = bronze_menstrual_cycle_diet_df[
    bronze_menstrual_cycle_diet_df['Cycle Start Date'] != '2077-08-10 20:52:00']

In [557]:
sorted(bronze_menstrual_cycle_startdate_df['Cycle Start Date'])

[Timestamp('2023-03-20 20:52:00'),
 Timestamp('2023-03-22 20:52:00'),
 Timestamp('2023-03-25 20:52:00'),
 Timestamp('2023-03-29 20:52:00'),
 Timestamp('2023-04-16 20:52:00'),
 Timestamp('2023-04-16 20:52:00'),
 Timestamp('2023-04-17 20:52:00'),
 Timestamp('2023-04-21 20:52:00'),
 Timestamp('2023-04-22 20:52:00'),
 Timestamp('2023-04-24 20:52:00'),
 Timestamp('2023-04-29 20:52:00'),
 Timestamp('2023-05-01 20:52:00'),
 Timestamp('2023-05-01 20:52:00'),
 Timestamp('2023-05-04 20:52:00'),
 Timestamp('2023-05-06 20:52:00'),
 Timestamp('2023-05-12 20:52:00'),
 Timestamp('2023-05-17 20:52:00'),
 Timestamp('2023-05-22 20:52:00'),
 Timestamp('2023-05-23 20:52:00'),
 Timestamp('2023-05-28 20:52:00'),
 Timestamp('2023-05-29 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-04 20:52:00'),
 Timestamp('2023-06-06 20:52:00'),
 Timestamp('2023-06-09 20:52:00'),
 Timestamp('2023-06-10 20:52:00'),
 Timestamp('2023-06-21 20:52:00'),
 Timestamp('2023-06-

In [558]:
bronze_menstrual_cycle_startdate_df['Next Cycle Start Date'] = pd.to_datetime(bronze_menstrual_cycle_startdate_df['Next Cycle Start Date'], format='%d/%m/%Y %H:%M')

C:\Users\cadav\AppData\Local\Temp\ipykernel_11152\2543635064.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bronze_menstrual_cycle_startdate_df['Next Cycle Start Date'] = pd.to_datetime(bronze_menstrual_cycle_startdate_df['Next Cycle Start Date'], format='%d/%m/%Y %H:%M')


In [559]:
bronze_menstrual_cycle_startdate_df.head()

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
0,1.0,18.0,2.0,Moderate,54.0,Low Carb,2024-11-13 20:52:00,26,7.0,2024-12-09 20:52:00,Headache,29.28,2025-11-06 19:16:40.445350
1,1.0,18.0,2.0,Moderate,54.0,Low Carb,2024-12-09 20:52:00,32,5.0,2025-01-10 20:52:00,Fatigue,29.28,2025-11-06 19:16:40.445350
2,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-01-10 20:52:00,41,7.0,2025-02-20 20:52:00,Fatigue,29.28,2025-11-06 19:16:40.445350
3,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-02-20 20:52:00,27,3.0,2025-03-19 20:52:00,Fatigue,29.28,2025-11-06 19:16:40.445350
4,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-03-19 20:52:00,42,5.0,2025-04-30 20:52:00,Cramps,29.28,2025-11-06 19:16:40.445350


In [549]:
bronze_menstrual_cycle_startdate_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 862 entries, 0 to 894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   User ID                862 non-null    float64       
 1   Age                    862 non-null    float64       
 2   Stress Level           862 non-null    float64       
 3   Exercise Frequency     862 non-null    object        
 4   Sleep Hours            862 non-null    float64       
 5   Diet                   862 non-null    object        
 6   Cycle Start Date       862 non-null    datetime64[ns]
 7   Cycle Length           862 non-null    int64         
 8   Period Length          862 non-null    float64       
 9   Next Cycle Start Date  862 non-null    datetime64[ns]
 10  Symptoms               862 non-null    object        
 11  BMI                    862 non-null    float64       
 12  _BronzeTimestamp       862 non-null    datetime64[us]
dtypes: datetim

In [560]:
sorted(bronze_menstrual_cycle_startdate_df['Next Cycle Start Date'])

[Timestamp('2023-04-17 20:52:00'),
 Timestamp('2023-04-22 20:52:00'),
 Timestamp('2023-04-29 20:52:00'),
 Timestamp('2023-05-01 20:52:00'),
 Timestamp('2023-05-12 20:52:00'),
 Timestamp('2023-05-17 20:52:00'),
 Timestamp('2023-05-22 20:52:00'),
 Timestamp('2023-05-23 20:52:00'),
 Timestamp('2023-05-29 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-04 20:52:00'),
 Timestamp('2023-06-06 20:52:00'),
 Timestamp('2023-06-09 20:52:00'),
 Timestamp('2023-06-10 20:52:00'),
 Timestamp('2023-06-23 20:52:00'),
 Timestamp('2023-06-27 20:52:00'),
 Timestamp('2023-06-28 20:52:00'),
 Timestamp('2023-07-02 20:52:00'),
 Timestamp('2023-07-07 20:52:00'),
 Timestamp('2023-07-10 20:52:00'),
 Timestamp('2023-07-10 20:52:00'),
 Timestamp('2023-07-11 20:52:00'),
 Timestamp('2023-07-12 20:52:00'),
 Timestamp('2023-07-13 20:52:00'),
 Timestamp('2023-07-14 20:52:00'),
 Timestamp('2023-07-17 20:52:00'),
 Timestamp('2023-07-20 20:52:00'),
 Timestamp('2023-07-

In [561]:
bronze_menstrual_cycle_nextstartdate_df = bronze_menstrual_cycle_startdate_df[
    bronze_menstrual_cycle_startdate_df['Next Cycle Start Date'] != '2077-08-10 20:52:00']

In [562]:
sorted(bronze_menstrual_cycle_nextstartdate_df['Next Cycle Start Date'])

[Timestamp('2023-04-17 20:52:00'),
 Timestamp('2023-04-22 20:52:00'),
 Timestamp('2023-04-29 20:52:00'),
 Timestamp('2023-05-01 20:52:00'),
 Timestamp('2023-05-12 20:52:00'),
 Timestamp('2023-05-17 20:52:00'),
 Timestamp('2023-05-22 20:52:00'),
 Timestamp('2023-05-23 20:52:00'),
 Timestamp('2023-05-29 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-03 20:52:00'),
 Timestamp('2023-06-04 20:52:00'),
 Timestamp('2023-06-06 20:52:00'),
 Timestamp('2023-06-09 20:52:00'),
 Timestamp('2023-06-10 20:52:00'),
 Timestamp('2023-06-23 20:52:00'),
 Timestamp('2023-06-27 20:52:00'),
 Timestamp('2023-06-28 20:52:00'),
 Timestamp('2023-07-02 20:52:00'),
 Timestamp('2023-07-07 20:52:00'),
 Timestamp('2023-07-10 20:52:00'),
 Timestamp('2023-07-10 20:52:00'),
 Timestamp('2023-07-11 20:52:00'),
 Timestamp('2023-07-12 20:52:00'),
 Timestamp('2023-07-13 20:52:00'),
 Timestamp('2023-07-14 20:52:00'),
 Timestamp('2023-07-17 20:52:00'),
 Timestamp('2023-07-20 20:52:00'),
 Timestamp('2023-07-

In [553]:
bronze_menstrual_cycle_nextstartdate_df.head()

,User ID,Age,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms,BMI,_BronzeTimestamp
0,1.0,18.0,2.0,Moderate,54.0,Low Carb,2024-11-13 20:52:00,26,7.0,2024-12-09 20:52:00,Headache,29.28,2025-11-06 19:16:40.445350
1,1.0,18.0,2.0,Moderate,54.0,Low Carb,2024-12-09 20:52:00,32,5.0,2025-01-10 20:52:00,Fatigue,29.28,2025-11-06 19:16:40.445350
2,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-01-10 20:52:00,41,7.0,2025-02-20 20:52:00,Fatigue,29.28,2025-11-06 19:16:40.445350
3,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-02-20 20:52:00,27,3.0,2025-03-19 20:52:00,Fatigue,29.28,2025-11-06 19:16:40.445350
4,1.0,18.0,2.0,Moderate,54.0,Low Carb,2025-03-19 20:52:00,42,5.0,2025-04-30 20:52:00,Cramps,29.28,2025-11-06 19:16:40.445350


## Guardar

In [554]:
os.makedirs(silver_path, exist_ok=True)

In [563]:
bronze_menstrual_cycle_nextstartdate_df.to_parquet(silver_file_path)